# Model Comparison

This notebook compares K-Means against Agglomerative (hierarchical) clustering and DBSCAN. It evaluates each method on silhouette score, cluster balance, and visual separation to determine the best approach for customer segmentation.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)

## Constants

In [2]:
str_bucket = 'cluster-analysis-demo'
str_uri = f's3://{str_bucket}/02_preprocessing/df_customers.parquet'

list_features = [
    'recency', 'frequency', 'monetary', 'avg_order_value',
    'avg_items_per_order', 'unique_products', 'avg_unit_price', 'tenure_days',
]

## Load Data and Preprocessing Pipeline

In [3]:
# load customer data
df = pd.read_parquet(str_uri)
print(f'Customers: {df.shape[0]:,}')

# load preprocessing pipeline from 03_clustering
cls_model_preprocessing = joblib.load('../03_clustering/output/cls_model_preprocessing.joblib')

# transform features
X = df[list_features].copy()
X = cls_model_preprocessing.transform(X)
print(f'Features: {X.shape[1]}')
X.head()

Customers: 4,312


AttributeError: Can't get attribute 'PreprocessingModel' on <module '__main__'>

## Load K-Means Results

In [ ]:
# load K-Means model
cls_kmeans = joblib.load('../03_clustering/output/cls_model_inference.joblib')
int_k = cls_kmeans.n_clusters
labels_kmeans = cls_kmeans.labels_
flt_sil_kmeans = silhouette_score(X, labels_kmeans)

print(f'K-Means: K={int_k}, Silhouette={flt_sil_kmeans:.4f}')
print(f'Cluster sizes: {np.bincount(labels_kmeans)}')

## Fit Agglomerative Clustering

In [ ]:
# use same K as K-Means for fair comparison
cls_agg = AgglomerativeClustering(n_clusters=int_k, linkage='ward')
labels_agg = cls_agg.fit_predict(X)
flt_sil_agg = silhouette_score(X, labels_agg)

print(f'Agglomerative (Ward): K={int_k}, Silhouette={flt_sil_agg:.4f}')
print(f'Cluster sizes: {np.bincount(labels_agg)}')

## Fit DBSCAN

In [ ]:
# tune eps using a range of values
list_eps = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
list_dbscan_results = []

for eps in list_eps:
    cls_db = DBSCAN(eps=eps, min_samples=10)
    labels_db = cls_db.fit_predict(X)
    int_n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
    int_n_noise = (labels_db == -1).sum()
    flt_noise_pct = int_n_noise / len(labels_db)
    flt_sil = silhouette_score(X, labels_db) if int_n_clusters >= 2 else -1
    list_dbscan_results.append({
        'eps': eps,
        'n_clusters': int_n_clusters,
        'n_noise': int_n_noise,
        'noise_pct': flt_noise_pct,
        'silhouette': flt_sil,
    })
    print(f'eps={eps}: {int_n_clusters} clusters, {int_n_noise} noise ({flt_noise_pct:.2%}), silhouette={flt_sil:.4f}')

df_dbscan = pd.DataFrame(list_dbscan_results)

In [ ]:
# select best DBSCAN by silhouette (among those with >= 2 clusters)
df_dbscan_valid = df_dbscan[df_dbscan['silhouette'] > -1].copy()
if len(df_dbscan_valid) > 0:
    idx_best = df_dbscan_valid['silhouette'].idxmax()
    flt_best_eps = df_dbscan_valid.loc[idx_best, 'eps']
    cls_dbscan_best = DBSCAN(eps=flt_best_eps, min_samples=10)
    labels_dbscan = cls_dbscan_best.fit_predict(X)
    flt_sil_dbscan = silhouette_score(X, labels_dbscan)
    int_n_clusters_dbscan = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    print(f'Best DBSCAN: eps={flt_best_eps}, {int_n_clusters_dbscan} clusters, Silhouette={flt_sil_dbscan:.4f}')
else:
    labels_dbscan = None
    flt_sil_dbscan = -1
    print('DBSCAN did not produce valid clusters for any eps value')

## Head-to-Head Comparison

In [ ]:
list_dict_row = [
    {
        'model': 'K-Means',
        'n_clusters': int_k,
        'silhouette': flt_sil_kmeans,
        'noise_pct': 0.0,
        'max_cluster_pct': np.bincount(labels_kmeans).max() / len(labels_kmeans),
        'min_cluster_pct': np.bincount(labels_kmeans).min() / len(labels_kmeans),
    },
    {
        'model': 'Agglomerative',
        'n_clusters': int_k,
        'silhouette': flt_sil_agg,
        'noise_pct': 0.0,
        'max_cluster_pct': np.bincount(labels_agg).max() / len(labels_agg),
        'min_cluster_pct': np.bincount(labels_agg).min() / len(labels_agg),
    },
]
if labels_dbscan is not None:
    labels_dbscan_clean = labels_dbscan[labels_dbscan >= 0]
    list_dict_row.append({
        'model': 'DBSCAN',
        'n_clusters': int_n_clusters_dbscan,
        'silhouette': flt_sil_dbscan,
        'noise_pct': (labels_dbscan == -1).sum() / len(labels_dbscan),
        'max_cluster_pct': np.bincount(labels_dbscan_clean).max() / len(labels_dbscan) if len(labels_dbscan_clean) > 0 else 0,
        'min_cluster_pct': np.bincount(labels_dbscan_clean).min() / len(labels_dbscan) if len(labels_dbscan_clean) > 0 else 0,
    })

df_comparison = pd.DataFrame(list_dict_row)
df_comparison.to_csv(f'{str_dirname_output}/model_comparison.csv', index=False)
print(f'Saved comparison to {str_dirname_output}/model_comparison.csv')
df_comparison

## Silhouette Score Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Comparison', fontsize=14, y=1.02)

# silhouette score
bars = axes[0].bar(df_comparison['model'], df_comparison['silhouette'],
                    color=['tab:blue', 'tab:orange', 'tab:green'][:len(df_comparison)],
                    alpha=0.8, edgecolor='black', linewidth=0.5)
axes[0].set_title('Silhouette Score')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_ylim(0, df_comparison['silhouette'].max() * 1.25)
for bar, val in zip(bars, df_comparison['silhouette']):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

# cluster count
bars2 = axes[1].bar(df_comparison['model'], df_comparison['n_clusters'],
                     color=['tab:blue', 'tab:orange', 'tab:green'][:len(df_comparison)],
                     alpha=0.8, edgecolor='black', linewidth=0.5)
axes[1].set_title('Number of Clusters')
axes[1].set_ylabel('K')
axes[1].set_ylim(0, df_comparison['n_clusters'].max() * 1.3)
for bar, val in zip(bars2, df_comparison['n_clusters']):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                 f'{val}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{str_dirname_output}/metrics_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

## PCA Visualization (2D Projection)

Project the 8-dimensional feature space down to 2 dimensions using PCA to visually compare how each algorithm partitions the data.

In [ ]:
# PCA for 2D projection
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
flt_var_explained = pca.explained_variance_ratio_.sum()
print(f'Variance explained by 2 components: {flt_var_explained:.2%}')

# build plot list
list_plots = [
    ('K-Means', labels_kmeans),
    ('Agglomerative', labels_agg),
]
if labels_dbscan is not None:
    list_plots.append(('DBSCAN', labels_dbscan))

int_n_plots = len(list_plots)
fig, axes = plt.subplots(1, int_n_plots, figsize=(7 * int_n_plots, 6))
fig.suptitle(f'Cluster Assignments (PCA 2D, {flt_var_explained:.0%} Variance Explained)', fontsize=14, y=1.02)
if int_n_plots == 1:
    axes = [axes]

for idx, (str_name, labels) in enumerate(list_plots):
    ax = axes[idx]
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='Set2', alpha=0.5, s=5)
    ax.set_title(str_name)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    # legend
    list_unique = sorted(set(labels))
    for lbl in list_unique:
        label_name = f'Noise' if lbl == -1 else f'Cluster {lbl}'
        ax.scatter([], [], c=[plt.cm.Set2(lbl / max(1, max(list_unique)))], label=label_name, s=20)
    ax.legend(fontsize=8, loc='best')

plt.tight_layout()
plt.savefig(f'{str_dirname_output}/pca_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

## Dendrogram (Hierarchical Clustering)

In [ ]:
# sample for readability (dendrogram with full dataset is unreadable)
int_sample = min(500, len(X))
np.random.seed(42)
arr_idx = np.random.choice(len(X), size=int_sample, replace=False)
X_sample = X.iloc[arr_idx].values

Z = linkage(X_sample, method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_title(f'Dendrogram (Ward Linkage, n={int_sample} sample)')
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=8, ax=ax)
ax.set_xlabel('Cluster Size')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/dendrogram.png', bbox_inches='tight', dpi=150)
plt.show()

## Cluster Size Comparison

In [ ]:
list_all_labels = [
    ('K-Means', labels_kmeans),
    ('Agglomerative', labels_agg),
]
if labels_dbscan is not None:
    list_all_labels.append(('DBSCAN', labels_dbscan))

int_n_models = len(list_all_labels)
fig, axes = plt.subplots(1, int_n_models, figsize=(6 * int_n_models, 5))
fig.suptitle('Cluster Size Distribution by Model', fontsize=14, y=1.02)
if int_n_models == 1:
    axes = [axes]

for idx, (str_name, labels) in enumerate(list_all_labels):
    ax = axes[idx]
    ser_counts = pd.Series(labels).value_counts().sort_index()
    bars = ax.bar(ser_counts.index.astype(str), ser_counts.values, color='tab:blue', alpha=0.8,
                  edgecolor='black', linewidth=0.5)
    ax.set_title(str_name)
    ax.set_xlabel('Cluster')
    ax.set_ylabel('N Customers')
    for bar, val in zip(bars, ser_counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                f'{val:,}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{str_dirname_output}/cluster_sizes_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

## Conclusion

In [ ]:
# determine champion
str_champion = df_comparison.loc[df_comparison['silhouette'].idxmax(), 'model']
flt_champion_sil = df_comparison['silhouette'].max()

str_summary = f"""
Three clustering algorithms were evaluated on the same preprocessed customer feature matrix:

1. K-Means (K={int_k}): Silhouette = {flt_sil_kmeans:.4f}
2. Agglomerative Ward (K={int_k}): Silhouette = {flt_sil_agg:.4f}
3. DBSCAN (best eps): Silhouette = {flt_sil_dbscan:.4f}

{str_champion} achieves the highest silhouette score ({flt_champion_sil:.4f}),
indicating the tightest, most well-separated clusters.

K-Means is preferred for customer segmentation due to its interpretability,
scalability, and ability to assign new customers to segments at inference time.

{str_champion} is the champion!
"""
print(str_summary)